# Kaplan-Meier + log-rank (deskriptif)
        **tabular › survival** · notebook `kaplan_meier`

        **Kapan dipakai:** Langkah pertama pada data survival: kurva kelangsungan hidup per kelompok dan uji perbedaan antar kelompok.

        **Jebakan umum:** Tidak memperhitungkan banyak fitur sekaligus (univariat); tidak memprediksi individu.

**Data demo:** Rossi (432 narapidana; durasi sampai ditangkap lagi, 74% tersensor). **Tersensor** = pengamatan berakhir tanpa kejadian (mis. belum ditangkap lagi sampai akhir studi): informasinya 'bertahan setidaknya sampai waktu ini', bukan hilang.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular_extra/survival/rossi_train.csv"; TEST_PATH = ROOT / "data_latihan/tabular_extra/survival/rossi_test.csv"
TEST_LABELS = TEST_PATH.with_name(TEST_PATH.name.replace("_test.csv", "_test_labels.csv"))      # durasi/kejadian uji (hanya latihan)
DURATION, EVENT = "week", "arrest"
FEATURES = ['fin', 'age', 'race', 'wexp', 'mar', 'paro', 'prio']
MODEL_KEY = "kaplan_meier"; SEED = 42
OUT = out_dir("tabular", "survival", MODEL_KEY); set_seed(SEED)

## 2. Data: durasi, kejadian, dan sensor
**Jangan** memperlakukan baris tersensor sebagai 'tidak ada kejadian' lalu memakai regresi/klasifikasi biasa: itu membuang informasi dan bias.

In [ ]:
from types import SimpleNamespace
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH).merge(pd.read_csv(TEST_LABELS), on="id")
print(f"latih {len(train)} | uji {len(test)} | kejadian pada latih: {int(train[EVENT].sum())} ({train[EVENT].mean():.0%}), sisanya tersensor ({1 - train[EVENT].mean():.0%})")
km = KaplanMeierFitter().fit(train[DURATION], train[EVENT]); ax = km.plot_survival_function(figsize=(6, 3.2)); ax.set_title("Kurva kelangsungan hidup keseluruhan (Kaplan-Meier)")
plt.tight_layout(); plt.savefig(OUT / "km_overall.png", dpi=130); plt.show()
print("median kelangsungan hidup:", km.median_survival_time_)

## 3. Bandingkan kelompok: kurva Kaplan-Meier + uji log-rank
Uji log-rank: H0 = kedua kelompok punya kurva kelangsungan hidup yang sama. p kecil → berbeda.

In [ ]:
from lifelines.statistics import logrank_test, multivariate_logrank_test
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4)); res = {}
for ax, (name, grp) in zip(axes, {"fin (bantuan keuangan)": train["fin"], "wexp (pengalaman kerja)": train["wexp"], "usia <= 24": (train["age"] <= 24).astype(int)}.items()):
    for g_ in sorted(grp.unique()):
        m_ = grp == g_; KaplanMeierFitter().fit(train.loc[m_, DURATION], train.loc[m_, EVENT], label=f"{g_} (n={int(m_.sum())})").plot_survival_function(ax=ax, ci_show=True)
    lr = multivariate_logrank_test(train[DURATION], grp, train[EVENT]); res[name] = float(lr.p_value); ax.set_title(f"{name}\np log-rank = {lr.p_value:.3f}")
plt.tight_layout(); plt.savefig(OUT / "km_groups.png", dpi=130); plt.show(); display(pd.Series(res, name="p-value log-rank").round(4).to_frame())

In [ ]:
save_json({"logrank_p": res, "median_survival": float(km.median_survival_time_) if np.isfinite(km.median_survival_time_) else None}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Untuk beberapa fitur sekaligus dan prediksi individu: Cox PH atau AFT di folder ini.
- Kurva KM tanpa perbedaan signifikan pada sampel kecil belum membuktikan 'tidak ada efek' (kuasa uji rendah).